In [2]:
from spark_config import get_spark_session
from pyspark.ml.fpm import FPGrowth

spark = get_spark_session(app_name="GPGrowth")


26/05/08 17:57:02 WARN Utils: Your hostname, khanhdo-VMware-Virtual-Platform resolves to a loopback address: 127.0.1.1; using 192.168.118.128 instead (on interface ens33)
26/05/08 17:57:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 17:57:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
df_real_user_portfolio = spark.read.parquet("/home/khanhdo/Documents/project/bigdata_mining/data_processed/real_user_portfolios")
df_real_user_portfolio.show(5, truncate=False)

+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+-----------+
|user_address                              |token_address                             |balance              |tx_count|last_active        |is_contract|
+------------------------------------------+------------------------------------------+---------------------+--------+-------------------+-----------+
|0x00000000fff16d2b960b3ef2686e40ba99403835|0xdac17f958d2ee523a2206206994597c13d831ec7|71.203482            |2       |2026-04-28 01:37:59|false      |
|0x00000000fff16d2b960b3ef2686e40ba99403835|0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2|2.0000001514545227   |8       |2026-04-28 01:54:47|false      |
|0x00006566a3158a643cfa9dbcfa9495a1a00bd7ea|0xdac17f958d2ee523a2206206994597c13d831ec7|0.0010000000000000009|2       |2026-02-08 10:13:59|false      |
|0x0001915c9c3d2911bc91b327e2425a7daa6b1a34|0xdac17f958d2ee523a2206206994597c13d831ec7|2.98155

In [4]:
df_tokens = spark.read.csv("/home/khanhdo/Documents/project/bigdata_mining/data_processed/processed_tokens.csv", header=True, inferSchema=True)

In [10]:
import pyspark.sql.functions as F

df_portfolio = df_real_user_portfolio.alias("p").join(
    df_tokens.alias("t"), 
    F.col("p.token_address") == F.col("t.address"), 
    "inner"
).select("p.user_address", "t.symbol", "p.balance", "p.tx_count")

In [12]:
df_portfolio.show(5, truncate=False)

+------------------------------------------+------+---------------------+--------+
|user_address                              |symbol|balance              |tx_count|
+------------------------------------------+------+---------------------+--------+
|0x00000000fff16d2b960b3ef2686e40ba99403835|USDT  |71.203482            |2       |
|0x00000000fff16d2b960b3ef2686e40ba99403835|WETH  |2.0000001514545227   |8       |
|0x00006566a3158a643cfa9dbcfa9495a1a00bd7ea|USDT  |0.0010000000000000009|2       |
|0x0001915c9c3d2911bc91b327e2425a7daa6b1a34|USDT  |2.981555974335137E-19|34      |
|0x000191bbf5a21d8a85e402c94b5beed4765580d4|USDT  |22.103434            |1       |
+------------------------------------------+------+---------------------+--------+
only showing top 5 rows



In [ ]:
# Nhóm các token (symbol) theo từng user_address thành một mảng (basket/danh sách items)
df_baskets = df_portfolio.groupBy("user_address").agg(F.collect_set("symbol").alias("items"))
df_baskets.show(5, truncate=False)

+------------------------------------------+---------------------------------------------------+
|user_address                              |items                                              |
+------------------------------------------+---------------------------------------------------+
|0x0000000000000000000000000000000000000001|[ZEON, NOW, KAT, BCDT, WETH, SXUT, MFT, DENT, KYTE]|
|0x00000000000000000000000000000000000012dc|[NMR]                                              |
|0x0000000000000000000000000000000000001cc1|[NMR]                                              |
|0x0000000000000000000000000000000000002c62|[NMR]                                              |
|0x00000000000000000000000000000000000036d6|[NMR]                                              |
+------------------------------------------+---------------------------------------------------+
only showing top 5 rows



In [33]:
from pyspark.ml.fpm import FPGrowth

# Lọc chỉ lấy những ví cầm từ 2 token trở lên
df_baskets_filtered = df_baskets.filter(F.size(F.col("items")) >= 2)

fpGrowth = FPGrowth(itemsCol="items", minSupport=0.005, minConfidence=0.3)

# Training mô hình trên tập dữ liệu đã lọc
model = fpGrowth.fit(df_baskets_filtered)

# Hiển thị các itemsets phổ biến (Frequent Itemsets)
model.freqItemsets.orderBy("freq", ascending=False).show(30, truncate=False)

+------------------+-----+
|items             |freq |
+------------------+-----+
|[USDT]            |20846|
|[WETH]            |9760 |
|[LINK]            |9372 |
|[WBTC]            |7845 |
|[WETH, USDT]      |7331 |
|[WBTC, USDT]      |5891 |
|[LINK, USDT]      |4184 |
|[QNT]             |3023 |
|[WBTC, WETH]      |2374 |
|[QNT, LINK]       |2055 |
|[LDO]             |1678 |
|[MANA]            |1193 |
|[WBTC, WETH, USDT]|1065 |
|[CRO]             |994  |
|[BAT]             |946  |
|[SNX]             |920  |
|[BNB]             |912  |
|[ANKR]            |910  |
|[LDO, LINK]       |889  |
|[WBTC, LINK]      |870  |
|[LINK, WETH]      |826  |
|[CHZ]             |769  |
|[BNB, USDT]       |751  |
|[VXT]             |716  |
|[ZRX]             |698  |
|[DENT]            |668  |
|[LPT]             |642  |
|[HANDY]           |601  |
|[VXT, USDT]       |598  |
|[TEL]             |582  |
+------------------+-----+
only showing top 30 rows



In [34]:
model.associationRules.orderBy("confidence", ascending=False).show(20, truncate=False)

+------------------+----------+-------------------+------------------+--------------------+
|antecedent        |consequent|confidence         |lift              |support             |
+------------------+----------+-------------------+------------------+--------------------+
|[VXT]             |[USDT]    |0.835195530726257  |1.3098458930266486|0.01829137735906769 |
|[BNB]             |[USDT]    |0.8234649122807017 |1.2914486413313335|0.022971278255284007|
|[WETH]            |[USDT]    |0.7511270491803279 |1.1780004134535382|0.22423760438014254 |
|[WBTC]            |[USDT]    |0.7509241555130657 |1.177682213191435 |0.18019147829810664 |
|[QNT]             |[LINK]    |0.6797882897783658 |2.371352812390537 |0.06285749242957207 |
|[TET]             |[USDT]    |0.6559485530546624 |1.0287309817238834|0.006239867861621754|
|[WBTC, LINK, WETH]|[USDT]    |0.6384615384615384 |1.00130591369678  |0.005077539534456918|
|[HANDY]           |[USDT]    |0.5524126455906821 |0.8663545343133537|0.01015507